# Build a Personal Finance Agent — runnable notebook

This notebook is a Colab/Kaggle/Binder-friendly version of the **Build a Personal Finance Agent** project from the
[Python & Data Analysis course](https://github.com/abderrahim-lectures/python-data-analysis-course). It mirrors the
real, runnable example at [`examples/finance-agent/`](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/examples/finance-agent/) —
pandas for loading/cleaning a bank CSV export, plus a LangChain [`deepagents`](https://github.com/langchain-ai/deepagents) tool-calling agent for categorizing ambiguous transactions and explaining spending anomalies in plain English.

**Privacy note:** the CSV used here is entirely synthetic — fake dates, fake merchants, fake amounts. Never run this notebook against a real, unredacted bank export; see the project doc's privacy tip for why.

You'll need a free-tier API key from one of six providers — no credit card required for any of them. See the
[project doc](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/docs/projects/finance-agent/index.md#setup)
for where to get a key for each provider.

Run the cells in order.

In [ ]:
!pip install pandas deepagents langchain-openai python-dotenv

## Load the synthetic sample transactions

This cell writes the same synthetic `transactions.csv` bundled in `examples/finance-agent/` directly into the notebook's working directory, so this notebook has no dependency on a repo checkout.

In [ ]:
CSV_TEXT = '''
date,description,amount
2026-05-01,ACME CORP PAYROLL DIRECT DEPOSIT,3200.00
2026-05-01,RIVERSIDE APARTMENTS RENT MAY,-1450.00
2026-05-02,TRADER JOES #112,-64.28
2026-05-02,STARBUCKS STORE 4521,-6.75
2026-05-03,SHELL OIL 57492,-41.10
2026-05-03,NETFLIX.COM,-15.49
2026-05-04,CHIPOTLE ONLINE 0083,-13.40
2026-05-04,UBER TRIP HELP.UBER.COM,-18.25
2026-05-05,WHOLE FOODS MARKET,-88.02
2026-05-05,SQ *JOES COFFEE CART,-4.50
2026-05-06,PACIFIC GAS ELECTRIC,-96.40
2026-05-06,COMCAST XFINITY,-79.99
2026-05-07,AMAZON.COM*3R4TT9QP3,-34.99
2026-05-07,CVS PHARMACY #3391,-22.15
2026-05-08,LYFT RIDE FRI PM,-15.60
2026-05-08,MCDONALDS F4821,-9.80
2026-05-09,SPOTIFY USA,-11.99
2026-05-09,TARGET T-2210,-56.73
2026-05-10,TST* CORNER BISTRO,-52.40
2026-05-11,SAFEWAY STORE 1748,-71.55
2026-05-11,PAYPAL *MERCHXYZ123,-27.00
2026-05-12,BART CLIPPER RELOAD,-40.00
2026-05-12,CHEVRON 00934,-38.75
2026-05-13,DOORDASH*THAI PALACE,-31.20
2026-05-13,VENMO PAYMENT JSMITH,-45.00
2026-05-14,WALGREENS #5521,-18.60
2026-05-14,AMC THEATRES 08811,-24.00
2026-05-15,CITY WATER UTIL DIST,-52.10
2026-05-15,WM SUPERCENTER #1234,-63.44
2026-05-16,COSTCO WHSE #445,-142.87
2026-05-16,STARBUCKS STORE 1187,-5.95
2026-05-17,STEAM GAMES PURCHASE,-19.99
2026-05-17,UBER EATS,-27.80
2026-05-18,BEST BUY #0219,-1849.00
2026-05-18,SHELL OIL 20481,-36.20
2026-05-19,TICKETMASTER EVENT,-89.50
2026-05-19,TRADER JOES #112,-58.11
2026-05-20,MONTHLY MAINTENANCE FEE,-12.00
2026-05-20,IKEA US ONLINE,-210.34
2026-05-21,CHIPOTLE ONLINE 0091,-14.10
2026-05-21,LYFT RIDE MON AM,-12.40
2026-05-22,KAISER PERMANENTE COPAY,-35.00
2026-05-22,WHOLE FOODS MARKET,-92.60
2026-05-23,NETFLIX.COM,-15.49
2026-05-23,SQ *JOES COFFEE CART,-4.75
2026-05-24,AMZN MKTP US*1H8KX2LP2,-46.21
2026-05-24,ATM WITHDRAWAL FEE,-3.50
2026-05-25,TARGET T-2210,-38.90
2026-05-25,TST* CORNER BISTRO,-47.15
2026-05-26,SAFEWAY STORE 1748,-66.32
2026-05-26,UBER TRIP HELP.UBER.COM,-21.75
2026-05-27,SPOTIFY USA,-11.99
2026-05-27,CVS PHARMACY #3391,-16.40
2026-05-28,PACIFIC GAS ELECTRIC,-101.20
2026-05-28,COMCAST XFINITY,-79.99
2026-05-29,DOORDASH*SUSHI HOUSE,-38.60
2026-05-29,PAYPAL *MERCHXYZ456,-19.99
2026-05-30,DELTA AIR LINES,-486.20
2026-05-30,MARRIOTT HOTELS RENO,-612.75
2026-05-31,ACME CORP PAYROLL DIRECT DEPOSIT,3200.00
2026-06-01,RIVERSIDE APARTMENTS RENT JUN,-1450.00
2026-06-01,TRADER JOES #112,-61.47
2026-06-02,STARBUCKS STORE 4521,-6.25
2026-06-02,SHELL OIL 57492,-39.60
2026-06-03,CHIPOTLE ONLINE 0083,-12.90
2026-06-03,VENMO PAYMENT MDIAZ,-60.00
2026-06-04,WHOLE FOODS MARKET,-84.19
2026-06-04,SQ *JOES COFFEE CART,-5.10
2026-06-05,COMCAST XFINITY,-79.99
2026-06-05,PACIFIC GAS ELECTRIC,-88.75
2026-06-06,AMAZON.COM*7F2KP9LM1,-29.49
2026-06-06,CVS PHARMACY #3391,-19.85
2026-06-07,LYFT RIDE SAT PM,-16.90
2026-06-07,MCDONALDS F4821,-8.55
2026-06-08,NETFLIX.COM,-15.49
2026-06-08,TARGET T-2210,-49.62
2026-06-09,TST* CORNER BISTRO,-412.50
2026-06-10,SAFEWAY STORE 1748,-69.80
2026-06-10,PAYPAL *MERCHXYZ789,-33.40
2026-06-11,BART CLIPPER RELOAD,-40.00
2026-06-11,CHEVRON 00934,-37.15
2026-06-12,DOORDASH*THAI PALACE,-29.90
2026-06-12,SPOTIFY USA,-11.99
2026-06-13,WALGREENS #5521,-14.20
2026-06-13,AMC THEATRES 08811,-24.00
2026-06-14,CITY WATER UTIL DIST,-49.60
2026-06-14,WM SUPERCENTER #1234,-71.02
2026-06-15,ACME CORP PAYROLL DIRECT DEPOSIT,3200.00
2026-06-15,COSTCO WHSE #445,-158.36
2026-06-16,STARBUCKS STORE 1187,-6.40
2026-06-16,STEAM GAMES PURCHASE,-14.99
2026-06-17,UBER EATS,-24.55
2026-06-17,SHELL OIL 20481,-34.80
2026-06-18,TICKETMASTER EVENT,-64.00
2026-06-18,TRADER JOES #112,-59.72
2026-06-19,MONTHLY MAINTENANCE FEE,-12.00
2026-06-19,OVERDRAFT FEE,-35.00
2026-06-20,CHIPOTLE ONLINE 0091,-13.75
2026-06-20,LYFT RIDE MON AM,-11.90
2026-06-21,KAISER PERMANENTE COPAY,-35.00
2026-06-21,WHOLE FOODS MARKET,-90.14
2026-06-22,NETFLIX.COM,-15.49
2026-06-22,SQ *JOES COFFEE CART,-4.95
2026-06-23,AMZN MKTP US*3R4TT9QP3,-52.18
2026-06-23,ATM WITHDRAWAL FEE,-3.50
2026-06-24,TARGET T-2210,-41.33
2026-06-24,TST* CORNER BISTRO,-46.80
2026-06-25,SAFEWAY STORE 1748,-63.94
2026-06-25,UBER TRIP HELP.UBER.COM,-19.45
2026-06-26,SPOTIFY USA,-11.99
2026-06-26,CVS PHARMACY #3391,-21.60
2026-06-27,PACIFIC GAS ELECTRIC,-93.85
2026-06-27,COMCAST XFINITY,-79.99
2026-06-28,DOORDASH*SUSHI HOUSE,-36.20
2026-06-28,PAYPAL *MERCHXYZ321,-24.99
2026-06-29,ACME CORP FREELANCE BONUS,850.00
2026-06-30,ACME CORP PAYROLL DIRECT DEPOSIT,3200.00
'''

with open('transactions.csv', 'w') as f:
    f.write(CSV_TEXT.strip() + '\n')


## Pick a provider and enter your API key

**You're free to use whichever provider you like.** GitHub Models is the suggested default below since it needs no
separate signup (you already have a GitHub account), but Gemini, Groq, Mistral, Cerebras, and OpenRouter all have
workable free tiers too. `getpass` is used instead of a plain `input()` so the key never gets echoed to the notebook
output or saved into its cell history.

In [ ]:
import os
from getpass import getpass

# One of: "github" (default), "gemini", "groq", "mistral", "cerebras", "openrouter"
PROVIDER = "github"

ENV_VAR_BY_PROVIDER = {
    "github": "GITHUB_TOKEN",
    "gemini": "GOOGLE_API_KEY",
    "groq": "GROQ_API_KEY",
    "mistral": "MISTRAL_API_KEY",
    "cerebras": "CEREBRAS_API_KEY",
    "openrouter": "OPENROUTER_API_KEY",
}

if PROVIDER not in ENV_VAR_BY_PROVIDER:
    raise ValueError(f"Unknown PROVIDER '{PROVIDER}'. Choose one of: {', '.join(ENV_VAR_BY_PROVIDER)}")

env_var = ENV_VAR_BY_PROVIDER[PROVIDER]
os.environ[env_var] = getpass(f"Enter your {PROVIDER} API key (stored only in {env_var} for this session): ")

## Build the model

Every provider here is OpenAI-compatible or has its own small LangChain integration package — mirrors
`_build_*_model()` in `finance_agent.py`.

In [ ]:
def build_model(provider: str):
    if provider == "github":
        from langchain_openai import ChatOpenAI

        return ChatOpenAI(
            model="gpt-4o-mini",  # confirm this still has a free tier before relying on it
            api_key=os.environ["GITHUB_TOKEN"],
            base_url="https://models.github.ai/inference",
        )
    if provider == "gemini":
        %pip install -q langchain-google-genai
        from langchain_google_genai import ChatGoogleGenerativeAI

        return ChatGoogleGenerativeAI(
            model="gemini-3.5-flash",  # pinned, versioned model ID -- not a "-latest" alias
            google_api_key=os.environ["GOOGLE_API_KEY"],
        )
    if provider == "groq":
        %pip install -q langchain-groq
        from langchain_groq import ChatGroq

        return ChatGroq(model="llama-3.3-70b-versatile", api_key=os.environ["GROQ_API_KEY"])
    if provider == "mistral":
        %pip install -q langchain-mistralai
        from langchain_mistralai import ChatMistralAI

        return ChatMistralAI(model="mistral-small-latest", api_key=os.environ["MISTRAL_API_KEY"])
    if provider == "cerebras":
        from langchain_openai import ChatOpenAI

        return ChatOpenAI(
            model="llama-3.3-70b",
            api_key=os.environ["CEREBRAS_API_KEY"],
            base_url="https://api.cerebras.ai/v1",
        )
    if provider == "openrouter":
        from langchain_openai import ChatOpenAI

        return ChatOpenAI(
            model="meta-llama/llama-3.3-70b-instruct:free",
            api_key=os.environ["OPENROUTER_API_KEY"],
            base_url="https://openrouter.ai/api/v1",
        )
    raise ValueError(f"Unknown provider '{provider}'")


model = build_model(PROVIDER)
model

## Load and clean the transactions with pandas

Same shape as any real bank export: date, description, signed amount (expenses negative, income positive).

In [ ]:
import pandas as pd

df = pd.read_csv("transactions.csv", parse_dates=["date"])
df["description"] = df["description"].str.strip()
df = df.dropna(subset=["date", "description", "amount"]).sort_values("date").reset_index(drop=True)
df.head()

## Step 1: rule-based baseline categorizer

A dict of keyword -> category, matched against each description. Fast and free, but it can't handle payment-processor
prefixes (`SQ *`, `TST*`, `PAYPAL *`, `AMZN MKTP US*...`) or peer-to-peer transfers — those are left as `None`.

In [ ]:
RULES = {
    "payroll direct deposit": "Income", "freelance bonus": "Income", "rent": "Housing",
    "trader joes": "Groceries", "whole foods": "Groceries", "safeway": "Groceries", "costco whse": "Groceries",
    "starbucks": "Dining", "chipotle": "Dining", "mcdonalds": "Dining", "doordash": "Dining", "uber eats": "Dining",
    "uber trip": "Transport", "lyft": "Transport", "shell oil": "Transport", "chevron": "Transport", "bart clipper": "Transport",
    "pacific gas electric": "Utilities", "comcast xfinity": "Utilities", "city water util": "Utilities",
    "netflix.com": "Subscriptions", "spotify": "Subscriptions",
    "amc theatres": "Entertainment", "steam games": "Entertainment", "ticketmaster": "Entertainment",
    "target": "Shopping", "amazon.com": "Shopping", "best buy": "Shopping", "ikea": "Shopping",
    "cvs pharmacy": "Healthcare", "walgreens": "Healthcare", "kaiser permanente": "Healthcare",
    "delta air lines": "Travel", "marriott hotels": "Travel", "airbnb": "Travel",
    "overdraft fee": "Fees", "atm withdrawal fee": "Fees", "monthly maintenance fee": "Fees",
}


def categorize_rule_based(description: str):
    text = description.lower()
    for keyword, category in RULES.items():
        if keyword in text:
            return category
    return None


df["category"] = df["description"].apply(categorize_rule_based)
resolved = df["category"].notna().sum()
print(f"Rule-based pass: {resolved}/{len(df)} categorized. {len(df) - resolved} left ambiguous.")
df[df["category"].isna()][["date", "description", "amount"]]

## Step 2 & 3: hand the ambiguous ones to an LLM agent

`create_deep_agent` wires the model together with a `categorize_transaction` tool. The agent reads each ambiguous
description and decides the category itself, instead of a hand-written `if`/`elif` chain.

In [ ]:
from deepagents import create_deep_agent

CATEGORIES = [
    "Income", "Housing", "Groceries", "Dining", "Transport", "Utilities",
    "Subscriptions", "Entertainment", "Shopping", "Healthcare", "Travel", "Fees", "Other",
]


def categorize_transaction(description: str, amount: float) -> str:
    """Categorize one bank transaction the rule-based pass couldn't confidently label.

    `description` is the raw bank description string; `amount` is signed (negative = money out).
    Must return exactly one of: Income, Housing, Groceries, Dining, Transport, Utilities,
    Subscriptions, Entertainment, Shopping, Healthcare, Travel, Fees, Other.
    """
    text = description.lower()
    if text.startswith("sq *") or text.startswith("tst*") or "coffee" in text or "bistro" in text:
        return "Dining"
    if text.startswith("venmo") or text.startswith("paypal"):
        return "Other"
    if text.startswith("amzn mktp"):
        return "Shopping"
    return "Other"


agent = create_deep_agent(
    model=model,
    tools=[categorize_transaction],
    system_prompt=(
        "You are a personal finance assistant. When asked to categorize a transaction, "
        "call the categorize_transaction tool rather than guessing. When asked to explain "
        "spending anomalies, summarize them for a non-technical reader in a short, "
        "plain-English paragraph, inventing nothing beyond the numbers you were given."
    ),
)

unresolved = df[df["category"].isna()]
for idx, row in unresolved.iterrows():
    result = agent.invoke({"messages": [{"role": "user", "content": f"Categorize this transaction: description={row['description']!r}, amount={row['amount']}"}]})
    text = str(result["messages"][-1].content)
    match = next((c for c in CATEGORIES if c.lower() in text.lower()), "Other")
    df.at[idx, "category"] = match

df["category"].value_counts()

## Step 4: flag statistical anomalies and summarize them

A per-category z-score: how many standard deviations a transaction sits above *that category's own* typical spend,
then the same agent explains what it found in plain English.

In [ ]:
spend = df["amount"].where(df["amount"] < 0)
df["spend_abs"] = spend.abs()
stats = df.groupby("category")["spend_abs"].agg(["mean", "std"]).rename(columns={"mean": "category_mean", "std": "category_std"})
df = df.join(stats, on="category")
safe_std = df["category_std"].replace(0, pd.NA)
df["z_score"] = (df["spend_abs"] - df["category_mean"]) / safe_std
df["is_anomaly"] = (df["z_score"] >= 2.0).fillna(False)

flagged = df[df["is_anomaly"]].sort_values("z_score", ascending=False)
summary_lines = [
    f"- {row['date'].date()} | {row['description']} | ${row['spend_abs']:.2f} in {row['category']} "
    f"(category average: ${row['category_mean']:.2f}, z-score: {row['z_score']:.1f})"
    for _, row in flagged.iterrows()
]
anomaly_summary = "\n".join(summary_lines) if summary_lines else "No anomalies found."
print(anomaly_summary)

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": (
    "Here are transactions flagged as statistically unusual for their category "
    "(z-score = how many standard deviations above that category's average spend):\n\n"
    f"{anomaly_summary}\n\n"
    "Summarize this for someone reviewing their bank statement, in 2-4 plain-English sentences. "
    "No new numbers, no advice beyond what the data supports."
)}]})
print(result["messages"][-1].content)